# Returns analysis

Handed over by a placement student who left in July. The last thing they wrote in
the handover email was that the model gets 0.94 and is ready to show the client.

Nobody has run this since.

In [34]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [35]:
df = pd.read_csv('/Users/kuberapaul/Downloads/lab_week01/data/retail_orders_week1.csv')
df.shape

(2030, 17)

In [36]:
df.head()

,order_id,order_date,customer_id,channel,category,item_price,quantity,discount_pct,order_value,payment_method,delivery_days,customer_region,customer_tenure_days,customer_prior_orders,marketing_opt_in,export_seq,returned
0,ORD-100000,2025-06-20,10417,web,apparel,41.47,1,0,41.47,bnpl,4,North,460,1,True,1705,0
1,ORD-100001,2025-12-22,10427,web,beauty,30.41,1,0,30.41,paypal,2,North,67,2,True,874,0
2,ORD-100002,2025-09-13,10570,web,electronics,234.17,1,0,234.17,card,1,Wales,583,1,True,1611,0
3,ORD-100003,2025-01-09,10535,web,beauty,13.06,1,0,13.06,card,4,London,750,0,True,933,0
4,ORD-100004,2025-12-09,10369,web,apparel,39.69,3,15,101.21,card,4,London,453,2,True,48,1


In [37]:
df.columns

Index(['order_id', 'order_date', 'customer_id', 'channel', 'category',
       'item_price', 'quantity', 'discount_pct', 'order_value',
       'payment_method', 'delivery_days', 'customer_region',
       'customer_tenure_days', 'customer_prior_orders', 'marketing_opt_in',
       'export_seq', 'returned'],
      dtype='str')

In [38]:
# drop the rows that look wrong, and put VAT on the order value
df = pd.read_csv('/Users/kuberapaul/Downloads/lab_week01/data/retail_orders_week1.csv')
df = df[df['quantity'] > 0]
df = df[df['delivery_days'] > 0]
df['order_value'] = df['order_value'] * 1.2
len(df)

2021

In [39]:

df['order_value'].head(1) # 41.47
# now run the next line twice, checking after each run
df['order_value'] = df['order_value'] * 1.2
df['order_value'].head(1)

0    59.7168
Name: order_value, dtype: float64

In [40]:
sample = df.sample(1200, random_state=9)
sample['returned'].mean()

np.float64(0.19083333333333333)

In [41]:
features = [ 'item_price', 'discount_pct', 'quantity',
            'delivery_'
            'days', 'customer_prior_orders']

X = df[features]
y = df['returned']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print('ROC-AUC', round(auc, 3))

ROC-AUC 0.587


In [43]:
coefs = pd.Series(model.coef_[0], index=features).sort_values()
coefs

customer_prior_orders   -0.019009
delivery_days            0.021380
item_price               0.034294
discount_pct             0.048160
quantity                 0.062325
dtype: float64

In [44]:

summary = (
    df.groupby('category', as_index=False)['returned']
      .mean()
      .rename(columns={'returned': 'return_rate'})
      .sort_values('return_rate', ascending=False)
)
summary

,category,return_rate
3,footwear,0.292683
0,apparel,0.238462
2,electronics,0.203762
5,sports,0.129412
4,home,0.111959
1,beauty,0.079208


0.94 is good enough. Send to client Friday.